# D0: baseline central da DeepResUNet

Este notebook treina a **DeepResUNet sem melhorias**. D0 será o ponto zero para pré-processamento, augmentation, transfer learning, fine-tuning e SAM.

Somente treino e validação são acessados. Teste interno e FUSeg permanecem fechados.

## Mapa da etapa

1. conferir configuração e casos visuais fixos;
2. fazer preflight de memória e gradientes;
3. treinar com checkpoint e retomada;
4. congelar a melhor época pelo Dice macro em threshold 0,5;
5. calibrar threshold usando somente a validação;
6. gerar curvas, métricas por imagem e painel fixo.

## 1. Montar o Drive e localizar o projeto

In [ ]:
from pathlib import Path
import os

try:
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    print("Execução fora do Colab.")

candidates = [
    Path("/content/drive/MyDrive/TCC/Projeto_TCC_Lucas"),
    Path("/content/drive/MyDrive/Projeto_TCC_Lucas"),
]
PROJECT_ROOT = next((p for p in candidates if p.is_dir()), Path.cwd())
if not (PROJECT_ROOT / "configs/baseline_d0.yaml").is_file():
    raise FileNotFoundError("Configuração do D0 não encontrada.")
os.chdir(PROJECT_ROOT)
print("Projeto:", PROJECT_ROOT)

## 2. Instalar dependências registradas

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"],
    text=True,
    capture_output=True,
)
print(result.stdout)
print(result.stderr)
result.check_returncode()

## 3. Usar cache temporário dos dados

Os resultados continuam no Drive. Apenas a leitura das imagens é feita no disco temporário para reduzir latência.

In [ ]:
import shutil

DATA_ROOT = Path("/content/tcc_data") if Path("/content").is_dir() else PROJECT_ROOT
if DATA_ROOT != PROJECT_ROOT:
    shutil.copytree(PROJECT_ROOT / "Dataset", DATA_ROOT / "Dataset", dirs_exist_ok=True)
print("Dados:", DATA_ROOT / "Dataset")

## 4. Importar funções e exibir a configuração congelada

In [ ]:
import csv, inspect, json, time, yaml
import matplotlib.pyplot as plt
import pandas as pd
import torch
from PIL import Image
from IPython.display import display

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from tcc_segmentation.data import DevelopmentDataset, ResizePad, create_development_dataloader
from tcc_segmentation.baseline_training import (
    aggregate_rows,
    fixed_subset,
    make_model,
    preflight_batch_size,
    save_fixed_validation_panel,
    subgroup_summary,
    sweep_validation_thresholds,
    thresholds_from_config,
    train_d0,
    write_csv,
    write_json,
)
from tcc_segmentation.models import DeepResUNet, count_parameters
from tcc_segmentation.training import load_training_checkpoint, seed_everything

CONFIG = yaml.safe_load((PROJECT_ROOT / "configs/baseline_d0.yaml").read_text(encoding="utf-8"))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
display(CONFIG)
print("Device:", DEVICE)

## 5. Criar treino e validação protegidos

Os manifestos têm hashes aprovados. A classe rejeita nomes de teste, avaliação externa e datasets diferentes do DFUC2022.

In [ ]:
transform = ResizePad(CONFIG["data"]["height"], CONFIG["data"]["width"])
train_dataset = DevelopmentDataset(
    PROJECT_ROOT / CONFIG["data"]["train_manifest"],
    DATA_ROOT,
    "train",
    transform=transform,
    mask_threshold=CONFIG["data"]["mask_threshold"],
)
validation_dataset = DevelopmentDataset(
    PROJECT_ROOT / CONFIG["data"]["validation_manifest"],
    DATA_ROOT,
    "validation",
    transform=transform,
    mask_threshold=CONFIG["data"]["mask_threshold"],
)
split_summary = json.loads((PROJECT_ROOT / "manifests/splits/split_summary.json").read_text())
MANIFEST_HASHES = {k: split_summary["manifest_hashes"][k] for k in ("train", "validation")}
display(
    {
        "treino": len(train_dataset),
        "validação": len(validation_dataset),
        "teste_acessado": False,
        "FUSeg_acessado": False,
    }
)

## 6. Visualizar os casos fixados antes do modelo

Eles foram selecionados por tamanho da ferida e número de componentes. A seleção não conhece nenhuma predição.

In [ ]:
visual_dataset = fixed_subset(validation_dataset, CONFIG["visualization"]["fixed_validation_ids"])
fig, axes = plt.subplots(len(visual_dataset), 2, figsize=(10, 3 * len(visual_dataset)))
for row in range(len(visual_dataset)):
    sample = visual_dataset[row]
    axes[row, 0].imshow(sample["image"].transpose(1, 2, 0))
    axes[row, 0].set_title(sample["canonical_id"] + " — imagem")
    axes[row, 1].imshow(sample["mask"][0], cmap="gray", vmin=0, vmax=1)
    axes[row, 1].set_title("referência")
    axes[row, 0].axis("off")
    axes[row, 1].axis("off")
plt.tight_layout()
plt.show()

## 7. Conferir a rede central e preparar a execução

Para retomar após uma desconexão, substitua `RESUME_DIR = None` pelo caminho da execução já criada e execute novamente a partir desta célula.

In [ ]:
model_preview = make_model(CONFIG)
print(model_preview)
print("Parâmetros:", count_parameters(model_preview))
assert isinstance(model_preview, DeepResUNet)
del model_preview

# Exemplo: RESUME_DIR = PROJECT_ROOT / 'results/day5/D0/D0_seed42_...'
RESUME_DIR = None
if RESUME_DIR is None:
    RUN_ID = time.strftime("D0_seed42_%Y%m%dT%H%M%SZ", time.gmtime())
    OUTPUT_DIR = PROJECT_ROOT / CONFIG["runtime"]["output_root"] / RUN_ID
    OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
    shutil.copy2(PROJECT_ROOT / "configs/baseline_d0.yaml", OUTPUT_DIR / "config_used.yaml")
    RESUME = False
else:
    OUTPUT_DIR = Path(RESUME_DIR)
    RUN_ID = OUTPUT_DIR.name
    RESUME = True
print("Execução:", RUN_ID)
print("Saídas:", OUTPUT_DIR)

## 8. Preflight de memória, logits, loss e gradientes

O modelo deste teste será descartado. Se batch 4 causar OOM, batch 2 será selecionado antes da época 1. Nenhuma métrica decide o batch.

In [ ]:
if not RESUME:
    SELECTED_BATCH, PREFLIGHT = preflight_batch_size(
        train_dataset=train_dataset, config=CONFIG, device=DEVICE
    )
    write_json(OUTPUT_DIR / "preflight.json", PREFLIGHT)
    display(PREFLIGHT)
else:
    PREFLIGHT = json.loads((OUTPUT_DIR / "preflight.json").read_text())
    SELECTED_BATCH = PREFLIGHT["selected_batch_size"]
    display({"retomada": True, "batch": SELECTED_BATCH})
write_json(
    OUTPUT_DIR / "environment_report.json",
    {
        "device": str(DEVICE),
        "torch_version": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_device_name": torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else None,
        "train_records": len(train_dataset),
        "validation_records": len(validation_dataset),
        "protected_partitions_accessed": False,
    },
)

## 9. Treinar D0

Cada linha impressa corresponde a uma época. `best_checkpoint.pt` guarda o maior Dice macro em 0,5; `last_checkpoint.pt` permite retomada.

In [ ]:
TRAINING_SUMMARY = train_d0(
    train_dataset=train_dataset,
    validation_dataset=validation_dataset,
    config=CONFIG,
    output_dir=OUTPUT_DIR,
    run_id=RUN_ID,
    device=DEVICE,
    manifest_hashes=MANIFEST_HASHES,
    resume=RESUME,
    preflight_result=None if RESUME else (SELECTED_BATCH, PREFLIGHT),
)
display(TRAINING_SUMMARY)

## 10. Curvas de treinamento e melhor época

In [ ]:
history_df = pd.read_csv(OUTPUT_DIR / "history.csv")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(history_df.epoch, history_df.train_loss, label="treino")
axes[0].plot(history_df.epoch, history_df.validation_loss, label="validação")
axes[0].set(title="Loss", xlabel="Época", ylabel="Loss")
axes[0].legend()
axes[1].plot(history_df.epoch, history_df.validation_dice_at_0_5)
axes[1].axvline(TRAINING_SUMMARY["best_epoch"], color="black", linestyle="--", label="melhor")
axes[1].set(title="Dice macro de validação em 0,5", xlabel="Época", ylabel="Dice")
axes[1].legend()
axes[2].plot(history_df.epoch, history_df.learning_rate)
axes[2].set(title="Learning rate", xlabel="Época", ylabel="LR")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=170)
plt.show()

## 11. Recarregar a melhor época e calibrar o threshold

Com o checkpoint fixado, a busca em grade usa os mesmos mapas de probabilidade da validação para selecionar o limiar com maior Dice macro.

In [ ]:
seed_everything(CONFIG["seed"])
model = make_model(CONFIG).to(DEVICE)
best_payload = load_training_checkpoint(
    OUTPUT_DIR / "best_checkpoint.pt", model=model, map_location=DEVICE, restore_rng=False
)
validation_loader = create_development_dataloader(
    validation_dataset,
    TRAINING_SUMMARY["selected_batch_size"],
    False,
    CONFIG["seed"],
    CONFIG["training"]["num_workers"],
)
metadata_by_id = {row["canonical_id"]: row for row in validation_dataset.rows}
sweep = sweep_validation_thresholds(
    model=model,
    loader=validation_loader,
    device=DEVICE,
    thresholds=thresholds_from_config(CONFIG),
    metadata_by_id=metadata_by_id,
)
write_csv(OUTPUT_DIR / "threshold_curve.csv", sweep["curve"])
write_csv(OUTPUT_DIR / "validation_per_image_at_0_5.csv", sweep["per_image_at_0_5"])
write_csv(OUTPUT_DIR / "validation_per_image_selected.csv", sweep["per_image_selected"])
metrics_05 = aggregate_rows(sweep["per_image_at_0_5"])
metrics_selected = aggregate_rows(sweep["per_image_selected"])
write_json(OUTPUT_DIR / "validation_metrics_at_0_5.json", metrics_05)
write_json(OUTPUT_DIR / "validation_metrics_selected.json", metrics_selected)
write_csv(OUTPUT_DIR / "validation_by_area_bin.csv", subgroup_summary(sweep["per_image_selected"]))
selection = {
    "checkpoint_epoch": int(best_payload["epoch"]),
    "checkpoint_rule": "highest_validation_macro_dice_at_threshold_0.5",
    "threshold": sweep["selected_threshold"],
    "threshold_rule": "highest_validation_macro_dice_then_closest_to_0.5",
    "grid": CONFIG["threshold_selection"],
    "metrics_at_0_5": metrics_05,
    "metrics_selected": metrics_selected,
    "protected_partitions_accessed": False,
}
write_json(OUTPUT_DIR / "threshold_selection.json", selection)
display(
    {
        "melhor_época": best_payload["epoch"],
        "threshold": sweep["selected_threshold"],
        "Dice_0.5": metrics_05["macro"]["dice"]["mean"],
        "Dice_selecionado": metrics_selected["macro"]["dice"]["mean"],
    }
)

## 12. Curvas de threshold

In [ ]:
curve_df = pd.DataFrame(sweep["curve"])
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
curve_df.plot(x="threshold", y=["dice", "iou"], ax=axes[0])
curve_df.plot(x="threshold", y=["precision", "recall"], ax=axes[1])
for axis in axes:
    axis.axvline(sweep["selected_threshold"], color="black", linestyle="--", label="selecionado")
    axis.legend()
axes[0].set_title("Sobreposição")
axes[1].set_title("Precisão e recall")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "threshold_curves.png", dpi=170)
plt.show()

## 13. Painel fixo e mapa de erros

No mapa de erros: verde = verdadeiro positivo, laranja = falso positivo e vermelho = falso negativo.

In [ ]:
panel_path = save_fixed_validation_panel(
    model=model,
    dataset=visual_dataset,
    device=DEVICE,
    threshold=sweep["selected_threshold"],
    output_path=OUTPUT_DIR / "fixed_validation_panel.png",
)
display(Image.open(panel_path))

## 14. Resumo final e arquivos para análise conjunta

In [ ]:
FINAL_SUMMARY = {
    "status": "completed",
    "scientific_role": "central_baseline_development",
    "run_id": RUN_ID,
    "seed": CONFIG["seed"],
    "model_parameters": count_parameters(model),
    "training": TRAINING_SUMMARY,
    "selection": selection,
    "partitions_accessed": ["train", "validation"],
    "protected_partitions_accessed": False,
}
write_json(OUTPUT_DIR / "final_summary.json", FINAL_SUMMARY)
display(FINAL_SUMMARY)
print("Arquivos:", OUTPUT_DIR)

## Não avance automaticamente

Arquive `final_summary.json`, `history.csv`, `training_curves.png`, `threshold_curve.csv`, `threshold_curves.png`, `threshold_selection.json`, `fixed_validation_panel.png` e `validation_by_area_bin.csv`.

Esses resultados devem ser interpretados antes de iniciar qualquer ablação. Teste interno e FUSeg continuam fechados.